# Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data

In [ ]:
project_root = Path.cwd().parent
raw_path = project_root / "data" / "spy_raw.csv"

df = pd.read_csv(raw_path, header=[0, 1], index_col=0)
df.columns = df.columns.get_level_values(0)
df.index.name = "Date"
df = df.reset_index()

df.head()

# Inspect data

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

# Convert date and sort

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.head()

# Create target

In [ ]:
df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df[["Date","Open",  "Close", "target"]].head(10)

# Create features

In [ ]:
df["return_1d"] = df["Close"].pct_change()
df["return_2d"] = df["Close"].pct_change(2)
df["return_5d"] = df["Close"].pct_change(5)
df["return_10d"] = df["Close"].pct_change(10)
df["return_20d"] = df["Close"].pct_change(20)

df["ma_5"] = df["Close"].rolling(5).mean()
df["ma_10"] = df["Close"].rolling(10).mean()
df["ma_20"] = df["Close"].rolling(20).mean()

df["ma_5_rel"] = df["Close"] / df["ma_5"] - 1
df["ma_10_rel"] = df["Close"] / df["ma_10"] - 1
df["ma_20_rel"] = df["Close"] / df["ma_20"] - 1

df["vol_5"] = df["return_1d"].rolling(5).std()
df["vol_20"] = df["return_1d"].rolling(20).std()
df["vol_ratio"] = df["vol_5"] / df["vol_20"]

df["volume_change_1d"] = df["Volume"].pct_change()

df["high_10"] = df["High"].rolling(10).max()
df["low_10"] = df["Low"].rolling(10).min()
df["range_pos_10"] = (df["Close"] - df["low_10"]) / (df["high_10"] - df["low_10"])

df["high_20"] = df["High"].rolling(20).max()
df["low_20"] = df["Low"].rolling(20).min()
df["range_pos_20"] = (df["Close"] - df["low_20"]) / (df["high_20"] - df["low_20"])

df["ma_spread_5_20"] = df["ma_5"] / df["ma_20"] - 1
df["ma_spread_10_20"] = df["ma_10"] / df["ma_20"] - 1

df["intraday_return"] = df["Close"] / df["Open"] - 1
df["high_low_range"] = df["High"] / df["Low"] - 1
df["close_to_high"] = df["Close"] / df["High"] - 1
df["close_to_low"] = df["Close"] / df["Low"] - 1

df = df.drop(columns=["ma_5", "ma_10", "ma_20", "high_10", "low_10", "high_20", "low_20"])

df.head()

# Inspect engineered features

In [ ]:
df.isna().sum()

# Drop missing values

In [ ]:
df = df.dropna().reset_index(drop=True)

df.head()

In [ ]:
df.shape

# Save final dataset

In [ ]:
features_path = project_root / "data" / "spy_features.csv"
df.to_csv(features_path, index=False)

print(features_path)